# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/medhu07/flyrankai/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

The goal of this lane is to rank content items for refresh review.

For the starter modeling exercise, I use the documented proxy label `is_declining_label`, defined as `trend_direction == "down"`. This is a current-window proxy rather than a future observed outcome, so it should not be interpreted as proof that a page will decline or that refreshing it will improve performance.

I will use a supervised classification model because the target is binary. The model's predicted probability of decline will be used as the ranking score, and performance will be evaluated using Precision@K.

I will start with a simple Logistic Regression model because it is interpretable. The model will be compared with the Week-4 rule-based baseline using the same evaluation split and metric.

In [2]:
!git clone https://github.com/medhu07/flyrankai.git
%cd flyrankai

Cloning into 'flyrankai'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 143 (delta 55), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.87 MiB | 4.80 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/flyrankai


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))


Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [7]:
# Create the documented starter proxy label.
# trend_direction is used only to construct the target,
# and will NOT be used as a model feature.

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


In [5]:
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).head(15))

Missing values:


,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468


In [6]:
!grep -RniE "target|label|outcome|proxy|declining|refresh|opportunity" \
    docs skills work \
    --exclude="*.ipynb" \
    --exclude="*.csv" \
    | head -100

docs/ml-intern-dataset-and-lane-guide.md:12:> This starter repo ships only the small anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
docs/ml-intern-dataset-and-lane-guide.md:25:- its verified committed outputs (`outputs/model_report.md`, `outputs/refresh_queue_sample.csv`, `outputs/charts/`); `outputs/model_results.json` is regenerated when you run the pipeline;
docs/ml-intern-dataset-and-lane-guide.md:26:- the small anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
docs/ml-intern-dataset-and-lane-guide.md:75:> **Building features and iterating?** Develop against one middle month (e.g. `month=2026-03`) and run the full fact table only for your final pass — hitting the full table over and over can trip Hugging Face rate limits (HTTP 429 errors). Avoid the `_sample` table for experiments: it holds the LAST month of data, and if your label is about "what happens next", experimenting there means peeking at the future you're supposed to predict.

## 2. Split design

I use a grouped train/test split by `client_id`, with 80% of clients assigned to training and 20% to testing.

Grouping by client prevents content from the same client appearing in both training and test sets. This gives a more honest estimate of how the model transfers to content from clients it did not see during training.

The split is fixed with a random seed so that the results are reproducible.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Fixed seed for reproducibility
RANDOM_STATE = 42

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print(
    "\nClient overlap:",
    len(
        set(train_df["client_id"]) &
        set(test_df["client_id"])
    )
)

print("\nTrain target rate:", train_df["is_declining_label"].mean())
print("Test target rate:", test_df["is_declining_label"].mean())

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Client overlap: 0

Train target rate: 0.5501111717078492
Test target rate: 0.5109524582184002


## 3. Train + compare vs my baseline

The model is a Logistic Regression classifier trained to predict the starter proxy label `is_declining_label`.

The model uses observable content, search, traffic, engagement, and freshness features available at the decision point. `trend_direction` and `trend_pct` are excluded because they are used to construct the proxy target and would leak target information into the model.

The model's predicted probability of decline is used as the ranking score. I compare it with the Week-4 rule-based baseline using Precision@K on the same held-out test clients.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

# Features that are NOT allowed to be used
excluded_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

feature_cols = [
    col for col in df.columns
    if col not in excluded_features
]

X_train = train_df[feature_cols]
y_train = train_df["is_declining_label"]

X_test = test_df[feature_cols]
y_test = test_df["is_declining_label"]

# Separate numeric and categorical columns
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Logistic Regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Probability of decline
test_df = test_df.copy()
test_df["model_probability"] = model.predict_proba(
    X_test
)[:, 1]

print("Model trained successfully.")

Numeric features: 29
Categorical features: 11
Model trained successfully.


In [10]:
# Evaluate ranking performance at K
def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()


# Model Precision@K
for k in [20, 50]:
    p_at_k = precision_at_k(
        y_test.reset_index(drop=True),
        test_df["model_probability"].reset_index(drop=True).values,
        k
    )
    print(f"Model Precision@{k}: {p_at_k:.3f}")

Model Precision@20: 1.000
Model Precision@50: 1.000


In [11]:
# Week-4 baseline on the same held-out test clients

impressions_threshold = train_df["impressions_90d"].median()
ctr_threshold = train_df["ctr"].median()
stale_threshold = 180

test_df["baseline_score"] = (
    (test_df["impressions_90d"] >= impressions_threshold).astype(int)
    + (test_df["ctr"] <= ctr_threshold).astype(int)
    + (test_df["days_since_last_update"] >= stale_threshold).astype(int)
)

for k in [20, 50]:
    p_at_k = precision_at_k(
        y_test.reset_index(drop=True),
        test_df["baseline_score"].reset_index(drop=True).values,
        k
    )
    print(f"Baseline Precision@{k}: {p_at_k:.3f}")

Baseline Precision@20: 0.800
Baseline Precision@50: 0.780


In [12]:
print("Test-set base rate:", y_test.mean())

Test-set base rate: 0.5109524582184002


### Comparison

On the held-out test clients, Logistic Regression achieved Precision@20 of 1.000 and Precision@50 of 1.000. The Week-4 rule-based baseline achieved Precision@20 of 0.800 and Precision@50 of 0.780. The test-set base rate was 0.511.

The model therefore ranked more positive examples at the top of the queue than the baseline on this test split for the starter proxy label.

These results should be interpreted as performance on the documented starter proxy label, not as evidence of future decline or evidence that a refresh will improve performance.

## 4. Errors and interpretation

The error analysis examines false positives and false negatives on the held-out test clients. I also inspect the model coefficients to identify which features are most strongly associated with the predicted outcome.

Because the target is a current-window proxy derived from `trend_direction`, unusually strong features will be checked for possible leakage or direct encoding of the target.

Three concrete errors will be reviewed to understand where the model is uncertain or wrong.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix, classification_report

# Convert probabilities to binary predictions using 0.50 threshold
test_df["model_prediction"] = (
    test_df["model_probability"] >= 0.50
).astype(int)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_test,
        test_df["model_prediction"]
    )
)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        test_df["model_prediction"],
        digits=3
    )
)

Confusion matrix:
[[2014 1000]
 [ 599 2550]]

Classification report:
              precision    recall  f1-score   support

           0      0.771     0.668     0.716      3014
           1      0.718     0.810     0.761      3149

    accuracy                          0.741      6163
   macro avg      0.745     0.739     0.739      6163
weighted avg      0.744     0.741     0.739      6163



In [14]:
# False positives: predicted decline, but proxy label is 0
false_positives = test_df[
    (test_df["model_prediction"] == 1) &
    (test_df["is_declining_label"] == 0)
].copy()

# False negatives: predicted non-decline, but proxy label is 1
false_negatives = test_df[
    (test_df["model_prediction"] == 0) &
    (test_df["is_declining_label"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(
    false_positives[
        [
            "content_id",
            "model_probability",
            "impressions_90d",
            "ctr",
            "days_since_last_update",
            "is_declining_label"
        ]
    ].head(3)
)

print("\nExample false negatives:")
display(
    false_negatives[
        [
            "content_id",
            "model_probability",
            "impressions_90d",
            "ctr",
            "days_since_last_update",
            "is_declining_label"
        ]
    ].head(3)
)

False positives: 1000
False negatives: 599

Example false positives:


,content_id,model_probability,impressions_90d,ctr,days_since_last_update,is_declining_label
13,content_a5a2fbc76336,0.652621,307,0.00,103,0
36,content_bce275871a25,0.626547,371,1.35,20,0
56,content_dcebfd222b10,0.536188,16,0.00,20,0



Example false negatives:


,content_id,model_probability,impressions_90d,ctr,days_since_last_update,is_declining_label
23,content_2da6ae9d0882,0.441060,297,0.34,20,1
25,content_033ae3e7aecf,0.495453,27,0.00,20,1
39,content_4595e8704e07,0.369989,4,0.00,104,1


In [16]:
# Get transformed feature names
preprocessor_fitted = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

feature_names = preprocessor_fitted.get_feature_names_out()
coefficients = classifier.coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values(
    "abs_coefficient",
    ascending=False
)

print("Top 15 features by absolute coefficient:")
display(coef_df.head(15))
print("Top features associated with higher predicted decline probability:")
display(
    coef_df.sort_values("coefficient", ascending=False)
    [["feature", "coefficient"]]
    .head(10)
)

print("Top features associated with lower predicted decline probability:")
display(
    coef_df.sort_values("coefficient", ascending=True)
    [["feature", "coefficient"]]
    .head(10)
)

Top 15 features by absolute coefficient:


,feature,coefficient,abs_coefficient
15,num__impressions_last_30d,-35.303416,35.303416
18,num__impressions_prev_30d,28.766050,28.766050
5,num__impressions_90d,2.170719,2.170719
70,cat__position_tier_top_3,-1.263342,1.263342
19,num__clicks_prev_30d,1.119618,1.119618
16,num__clicks_last_30d,-0.996492,0.996492
8,num__sessions_90d,0.628843,0.628843
37,cat__main_intent_navigational,-0.559768,0.559768
13,num__days_with_impressions,0.534706,0.534706
9,num__users_90d,-0.519732,0.519732


Top features associated with higher predicted decline probability:


,feature,coefficient
18,num__impressions_prev_30d,28.766050
5,num__impressions_90d,2.170719
19,num__clicks_prev_30d,1.119618
8,num__sessions_90d,0.628843
13,num__days_with_impressions,0.534706
44,cat__model_used_gpt-5-mini,0.444113
69,cat__position_tier_striking,0.373991
34,cat__content_type_keyword article,0.341666
66,cat__position_tier_deep,0.324880
7,num__pageviews_90d,0.315212


Top features associated with lower predicted decline probability:


,feature,coefficient
15,num__impressions_last_30d,-35.303416
70,cat__position_tier_top_3,-1.263342
16,num__clicks_last_30d,-0.996492
37,cat__main_intent_navigational,-0.559768
9,num__users_90d,-0.519732
21,num__content_age_days,-0.465313
51,cat__freshness_tier_181+,-0.459541
17,num__sessions_last_30d,-0.453229
33,cat__content_type_feedly article,-0.409042
41,cat__model_used_gemini-2.5-flash,-0.386866


The model made 1,000 false-positive predictions and 599 false-negative predictions on the held-out test clients. At a 0.50 classification threshold, the model achieved 74.1% accuracy, with 81.0% recall and 71.8% precision for the declining class.

The error examples show that the model can be uncertain for pages with different levels of visibility and freshness. For example, a false positive had a predicted probability of 0.653 with 307 impressions and 103 days since update, while another false positive had a probability of 0.627 despite being updated only 20 days ago. A false negative had a probability of 0.441 with 297 impressions and 20 days since update.

The largest absolute Logistic Regression coefficients were associated with `impressions_last_30d` (-35.30), `impressions_prev_30d` (+28.77), and `impressions_90d` (+2.17). These features are closely related to the current performance/trend information used to construct the proxy label, so their large coefficients are a reason for caution rather than evidence of causal importance.

The model therefore appears to rely strongly on recent and previous-period impression signals. Because the target is the current-window proxy `trend_direction == "down"`, this strong relationship is expected to some extent, but it also limits how much the result can tell us about predicting genuinely future decline.

In [17]:
# Confirm that target/leakage-prone fields are not model features
print("Excluded from model features:")
print(excluded_features)

print("\nTrend-derived columns present in model features:")
print([
    col for col in feature_cols
    if col in ["trend_direction", "trend_pct", "is_declining_label"]
])

Excluded from model features:
['content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label']

Trend-derived columns present in model features:
[]


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.